# Sends a completion request (Gemini Developer API)

In [ ]:
from google import genai
from google.genai import types

# Example of single API call directly
# See https://ai.google.dev/gemini-api/docs/migrate

client = genai.Client(api_key="XXXXXX")

# Optional config
config=types.GenerateContentConfig(
      system_instruction='you are a jocker and trying to trick with wrong answers',
      max_output_tokens= 400,
      top_k= 2,
      top_p= 0.5,
      temperature= 0.5,
      response_mime_type= 'application/json',
#      stop_sequences= ['\n'],
      seed=42,
   )

response = client.models.generate_content(
    model="gemini-2.0-flash-lite",
    config=config,
    contents="Solve x^2 = 4",
)

print(response.model_dump_json(
    exclude_none=True, indent=4))
print(response.text)

# Google cloud (bucket) access

Need to create a service account and generate/download key. 
In this example key is in llm-equivalence-4e33ecb345c8.json 


In [ ]:
from google.cloud import storage
storage_client = storage.Client.from_service_account_json('../llm_expr_equivalence/llm-equivalence-4e33ecb345c8.json')

bucket = storage_client.get_bucket("llm-equivalence-batch")
blob = bucket.blob("test_blob")
blob.upload_from_filename("../Data/openai/openai-batch-20250305-131406.jsonl")
blob.download_to_filename("tmp.download")

# Authenticate and send completion request (Vertex AI API)

The procedure is more complicated then Dev API. Need to use Oauth2 API. Many nontrivial steps including:
1. Configure the OAuth consent screen (in the Google Cloud console)
2. Authorize credentials for a desktop application (can be other applications) and download client_secret.json with client data
3. Convert client_secret.json to tokens acceptable by the API credentials. These expire so in app need to be able to refresh them.
   
Details are here: https://ai.google.dev/gemini-api/docs/oauth

Scopes list can be found here: https://developers.google.com/identity/protocols/oauth2/scopes

In [ ]:
# Create tokens file from client_secret_..._.json (downloaded using Google Cloud console)
from google_auth_oauthlib.flow import InstalledAppFlow

# Scopes list can be found here: 
# https://developers.google.com/identity/protocols/oauth2/scopes
SCOPES=['https://www.googleapis.com/auth/cloud-platform']

flow = InstalledAppFlow.from_client_secrets_file(
                '../llm_expr_equivalence/client_secret_8106924470-9lkf64u90keonuggfnkj79h62371gd38.apps.googleusercontent.com.json', SCOPES)
creds = flow.run_local_server(port=0)

# Save tokens file
with open('../llm_expr_equivalence/token.json', 'w') as token:
    token.write(creds.to_json())

In [ ]:
from google import genai
from google.genai.types import CreateBatchJobConfig, GenerateContentConfig
from google.oauth2.credentials import Credentials


creds = Credentials.from_authorized_user_file('../llm_expr_equivalence/token.json')
client = genai.Client(vertexai=True, project="llm-equivalence", location='us-central1', credentials=creds)



# INPUT_DATA = "gs://llm-equivalence-batch/test_blob"  
# BUCKET_URI = "gs://llm-equivalence-results"  # @param {type:"string"} output bucket
# gcs_batch_job = client.batches.create(
#    model="gemini-2.0-flash-lite",
#    src=INPUT_DATA,
#    config=CreateBatchJobConfig(dest=BUCKET_URI, display_name="llm-equivalence-batch"),
# )
# gcs_batch_job.name

response = client.models.generate_content(
    model="gemini-2.0-flash-lite",
    config=GenerateContentConfig(
      system_instruction='you are a jocker and trying to trick with wrong answers',
      max_output_tokens= 400,
      top_k= 2,
      top_p= 0.5,
      temperature= 0.5,
      response_mime_type= 'application/json',
#      stop_sequences= ['\n'],
      seed=42,
   ),
    contents="Solve x^2 = 4",
)
print(response.text)

# Batch prediction processing (Vertex AI API and bucket data storage)

**Note:** When using API, Google Cloud will create service agent accounts (i.e  *AI Platform Service Agent*, *Vertex AI Batch Prediction Service Agent*) with default perissions. They should have access read/write access to buckets, but I had to go and manually add Storage Admmin role to the service account fo the "results" bucket where batch prediction service ouptits the results.

### Batch input format
Description: https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/inference

Example: {"request":{"contents": [{"role": "user", "parts": [{"text": "List objects in this image."}, {"file_data": {"file_uri": "gs://cloud-samples-data/generative-ai/image/office-desk.jpeg", "mime_type": "image/jpeg"}}]}],"generationConfig":{"temperature": 0.4}}}

In [ ]:
from google import genai
from google.genai.types import CreateBatchJobConfig, GenerateContentConfig
from google.oauth2.credentials import Credentials


creds = Credentials.from_authorized_user_file('../llm_expr_equivalence/token.json')
client = genai.Client(vertexai=True, project="llm-equivalence", location='us-central1', credentials=creds)


INPUT_DATA = "gs://llm-equivalence-batch/test_blob"  
BUCKET_URI = "gs://llm-equivalence-results"  # @param {type:"string"} output bucket
gcs_batch_job = client.batches.create(
   model="gemini-2.0-flash-lite-001",
   src=INPUT_DATA,
   config=CreateBatchJobConfig(dest=BUCKET_URI, display_name="llm-equivalence-batch"),
)
gcs_batch_job.name


In [ ]:
gcs_batch_job = client.batches.get(name=gcs_batch_job.name)
gcs_batch_job

for job in client.batches.list():
    print(job.name, job.create_time, job.state)

# Read output data into Dataframe using filesystem module

In [92]:
import fsspec
import pandas as pd
from google.cloud import storage
from google.cloud.storage import Blob

fs = fsspec.filesystem("gcs")
batch_dest_uri = "gs://llm-equivalence-results"
file_paths = fs.glob(f"{batch_dest_uri}/*/predictions.jsonl")

if file_paths:
    file_name = file_paths[-1]
    print(file_name)

    storage_client = storage.Client.from_service_account_json('../llm_expr_equivalence/llm-equivalence-storage-creds.json')
    blob = Blob.from_string(f"gs://{file_name}", client=storage_client)
    blob.download_to_filename("/tmp/aaa.jsonl")

llm-equivalence-results/prediction-model-2025-04-25T18:22:04.203803Z/predictions.jsonl
